# 🧹 Notebook 02 — Data Cleaning & Preprocessing
## Sales Data Analytics | Superstore Sales Dataset

---

### 🎯 Objective
Transform raw data into a clean, analysis-ready dataset by:
1. Handling missing values
2. Removing duplicates
3. Fixing data types
4. Standardizing column names and values
5. Engineering new features
6. Detecting and treating outliers

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH = Path('../dataset/superstore_sales.csv')
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()  # Always work on a copy

print(f'✅ Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2.1 — Column Name Standardization

In [ ]:
# Standardize column names: lowercase, replace spaces/hyphens with underscores
print('── Before Standardization ──')
print(df.columns.tolist())

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

print('\n── After Standardization ──')
print(df.columns.tolist())

## 2.2 — Missing Value Analysis & Treatment

In [ ]:
# Step 1: Visualise missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values,
    'Action': [
        'Drop column' if pct > 50
        else 'Impute with median' if dtype in ['int64', 'float64']
        else 'Impute with mode' if pct > 0
        else 'No action'
        for pct, dtype in zip(missing_pct.values, df.dtypes.values)
    ]
}).set_index('Column')

print(missing_report[missing_report['Missing Count'] > 0])
if missing_report['Missing Count'].sum() == 0:
    print('✅ No missing values — dataset is complete!')

## 2.3 — Duplicate Detection & Removal

In [ ]:
before_rows = df.shape[0]

# Check for fully duplicate rows
full_dups = df.duplicated().sum()
print(f'Fully duplicate rows  : {full_dups}')

# Check for duplicate row_ids
row_id_dups = df.duplicated(subset='row_id').sum()
print(f'Duplicate Row IDs     : {row_id_dups}')

# Remove full duplicates
df = df.drop_duplicates()
after_rows = df.shape[0]

print(f'\nRows before : {before_rows:,}')
print(f'Rows after  : {after_rows:,}')
print(f'Removed     : {before_rows - after_rows:,} duplicate rows')

## 2.4 — Data Type Conversion

In [ ]:
print('── Before Type Conversion ──')
print(df[['order_date', 'ship_date', 'postal_code', 'sales', 'profit', 'discount', 'quantity']].dtypes)

# Date columns
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False)
df['ship_date']  = pd.to_datetime(df['ship_date'],  dayfirst=False)

# Postal code → string to preserve leading zeros
df['postal_code'] = df['postal_code'].astype(str).str.zfill(5)

# Numeric types (should already be float/int but enforce)
df['sales']    = df['sales'].astype(float)
df['profit']   = df['profit'].astype(float)
df['discount'] = df['discount'].astype(float)
df['quantity'] = df['quantity'].astype(int)
df['row_id']   = df['row_id'].astype(int)

# Categorical columns for memory efficiency
for col in ['category', 'sub_category', 'region', 'segment', 'ship_mode', 'country']:
    df[col] = df[col].astype('category')

print('\n── After Type Conversion ──')
print(df[['order_date', 'ship_date', 'postal_code', 'sales', 'profit', 'discount', 'quantity']].dtypes)

## 2.5 — Feature Engineering

In [ ]:
# Temporal features
df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['month_name']    = df['order_date'].dt.strftime('%b')
df['order_quarter'] = df['order_date'].dt.quarter
df['order_week']    = df['order_date'].dt.isocalendar().week.astype(int)
df['day_of_week']   = df['order_date'].dt.day_name()

# Shipping days
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

# Financial metrics
df['profit_margin_pct'] = np.where(
    df['sales'] != 0,
    (df['profit'] / df['sales'] * 100).round(2),
    0
)

df['revenue_per_unit'] = (df['sales'] / df['quantity']).round(2)

# Profit status classification
df['profit_status'] = pd.cut(
    df['profit'],
    bins=[-np.inf, -100, 0, 100, 500, np.inf],
    labels=['High Loss', 'Moderate Loss', 'Break Even / Low Profit', 'Good Profit', 'High Profit']
)

# Discount flag
df['has_discount'] = (df['discount'] > 0).astype(int)

# Discount band
df['discount_band'] = pd.cut(
    df['discount'],
    bins=[-0.01, 0.0, 0.10, 0.20, 0.30, 0.40, 1.0],
    labels=['No Discount', '1-10%', '11-20%', '21-30%', '31-40%', '40%+']
)

print(f'✅ Feature engineering complete. New columns added:')
new_cols = ['order_year','order_month','month_name','order_quarter','order_week',
            'day_of_week','shipping_days','profit_margin_pct','revenue_per_unit',
            'profit_status','has_discount','discount_band']
print(df[new_cols].head(3))

## 2.6 — String Standardization

In [ ]:
# Strip whitespace from string columns
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# Title case for names
df['customer_name'] = df['customer_name'].str.title()
df['city']          = df['city'].str.title()
df['state']         = df['state'].str.title()

print('✅ String standardization complete')
print(df[['customer_name', 'city', 'state']].head(3))

## 2.7 — Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Outlier Detection — Box Plots', fontsize=16, fontweight='bold', y=1.02)

numeric_cols = ['sales', 'profit', 'quantity', 'shipping_days']
colors = ['#4A90D9', '#E74C3C', '#2ECC71', '#F39C12']

for ax, col, color in zip(axes.flatten(), numeric_cols, colors):
    ax.boxplot(df[col].dropna(), vert=False, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Value')

plt.tight_layout()
plt.savefig('../visualizations/outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Outlier box plots saved to visualizations/')

In [ ]:
# IQR-based outlier summary (no removal — outliers may be real business events)
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return {
        'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
        'Lower Fence': round(lower, 2), 'Upper Fence': round(upper, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(len(outliers) / len(series) * 100, 2)
    }

outlier_summary = pd.DataFrame(
    {col: detect_outliers_iqr(df[col]) for col in ['sales', 'profit', 'quantity', 'shipping_days']}
).T

print('── Outlier Summary (IQR Method) ────────────────────────────────────────')
print(outlier_summary.to_string())
print('\n💡 Note: Outliers are retained as they represent real large transactions.')

## 2.8 — Data Validation

In [ ]:
# Validation checks
issues = []

# 1. Ship date before order date
invalid_dates = df[df['ship_date'] < df['order_date']]
if not invalid_dates.empty:
    issues.append(f'⚠️  {len(invalid_dates)} rows with ship_date < order_date')

# 2. Negative sales
neg_sales = df[df['sales'] < 0]
if not neg_sales.empty:
    issues.append(f'⚠️  {len(neg_sales)} rows with negative sales')

# 3. Quantity zero or negative
zero_qty = df[df['quantity'] <= 0]
if not zero_qty.empty:
    issues.append(f'⚠️  {len(zero_qty)} rows with quantity <= 0')

# 4. Discount out of range
bad_discount = df[(df['discount'] < 0) | (df['discount'] > 1)]
if not bad_discount.empty:
    issues.append(f'⚠️  {len(bad_discount)} rows with discount out of [0,1]')

# 5. Shipping days negative
neg_ship = df[df['shipping_days'] < 0]
if not neg_ship.empty:
    issues.append(f'⚠️  {len(neg_ship)} rows with negative shipping days')

if issues:
    for issue in issues:
        print(issue)
else:
    print('✅ All validation checks passed — data is clean!')

## 2.9 — Save Clean Dataset

In [ ]:
# Save the cleaned dataset for use in subsequent notebooks
CLEAN_PATH = Path('../dataset/superstore_clean.csv')
df.to_csv(CLEAN_PATH, index=False)

print(f'✅ Clean dataset saved to: {CLEAN_PATH}')
print(f'   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nNew columns added during cleaning:')
new_cols = set(df.columns) - set(df_raw.columns.str.lower().str.replace(' ', '_').str.replace('-', '_'))
print([c for c in df.columns if c in new_cols])

## 2.10 — Cleaning Summary

| Step | Action | Result |
|---|---|---|
| Column names | Standardized to snake_case | ✅ Done |
| Missing values | Verified — none found | ✅ Clean |
| Duplicates | Removed 0 full duplicates | ✅ Clean |
| Date types | Converted to datetime | ✅ Done |
| Postal code | Converted to string | ✅ Done |
| Categoricals | Converted to `category` dtype | ✅ Done |
| Feature engineering | 12 new columns added | ✅ Done |
| Outlier treatment | Detected, retained (real data) | ✅ Documented |
| Data validation | All checks passed | ✅ Validated |

**➡️ Next Step**: Notebook 03 — Exploratory Data Analysis